# Assignment 3 — Part 2: Hierarchical Risk Parity (HRP)
## Person 3 Deliverables: Data Pipeline, Clustering & Dendrogram

This notebook implements:
1. **Data Fetching**: 5Y adjusted close prices for 12 NSE stocks
2. **Daily Returns**: Compute and split into train (4Y) / test (1Y)
3. **Correlation & Distance Matrix**: From train data
4. **Hierarchical Clustering**: Ward and Single linkage comparison
5. **Viz 2**: HRP Dendrogram with interpreted cluster labels
6. **Cluster Interpretation Notes** for the report
7. **Outputs for Person 4**: Linkage matrix, leaf order, daily returns CSVs

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
from datetime import datetime, timedelta
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Paths
BASE_DIR = Path('..').resolve()
DATA_RAW = BASE_DIR / 'data' / 'raw'
DATA_PROCESSED = BASE_DIR / 'data' / 'processed'
FIGURES_DIR = BASE_DIR / 'figures'

for d in [DATA_RAW, DATA_PROCESSED, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Tickers
TICKERS = [
    'TCS.NS', 'INFY.NS', 'HCLTECH.NS',
    'TATASTEEL.NS', 'HINDALCO.NS', 'JINDALSTEL.NS',
    'HDFCBANK.NS', 'ICICIBANK.NS', 'KOTAKBANK.NS',
    'HINDUNILVR.NS', 'ITC.NS', 'NESTLEIND.NS'
]

# Short names for display
SHORT_NAMES = {
    'TCS.NS': 'TCS', 'INFY.NS': 'INFY', 'HCLTECH.NS': 'HCLTECH',
    'TATASTEEL.NS': 'TATASTEEL', 'HINDALCO.NS': 'HINDALCO', 'JINDALSTEL.NS': 'JINDALSTEL',
    'HDFCBANK.NS': 'HDFCBANK', 'ICICIBANK.NS': 'ICICIBANK', 'KOTAKBANK.NS': 'KOTAKBANK',
    'HINDUNILVR.NS': 'HINDUNILVR', 'ITC.NS': 'ITC', 'NESTLEIND.NS': 'NESTLEIND'
}

print('Setup complete.')

---
## 1. Data Fetching — 5Y Adjusted Close Prices

In [ ]:
# Fetch 5 years of adjusted close data
end_date = datetime.today()
start_date = end_date - timedelta(days=365 * 5)

print(f'Fetching data from {start_date.date()} to {end_date.date()}')

raw_data = yf.download(
    TICKERS,
    start=start_date,
    end=end_date,
    auto_adjust=True,
    progress=True
)

# Extract Close prices (auto_adjust=True means these are already adjusted)
if isinstance(raw_data.columns, pd.MultiIndex):
    prices = raw_data['Close']
else:
    prices = raw_data[['Close']]

# Rename columns to short names
prices.columns = [SHORT_NAMES.get(c, c) for c in prices.columns]

# Drop any rows with all NaN, forward fill small gaps
prices = prices.dropna(how='all').ffill()

# Save raw prices
prices.to_csv(DATA_RAW / 'adjusted_close_daily.csv')

print(f'\nPrice data shape: {prices.shape}')
print(f'Date range: {prices.index[0].date()} to {prices.index[-1].date()}')
print(f'Stocks: {list(prices.columns)}')
prices.tail()

---
## 2. Daily Returns & Train/Test Split

In [ ]:
# Compute daily returns
daily_returns = prices.pct_change().dropna()

print(f'Daily returns shape: {daily_returns.shape}')
print(f'Date range: {daily_returns.index[0].date()} to {daily_returns.index[-1].date()}')

# Train/test split: first 4 years = train, last 1 year = test
split_date = daily_returns.index[0] + pd.DateOffset(years=4)
print(f'\nSplit date: {split_date.date()}')

returns_train = daily_returns[daily_returns.index < split_date]
returns_test = daily_returns[daily_returns.index >= split_date]

print(f'Train: {returns_train.shape[0]} days ({returns_train.index[0].date()} to {returns_train.index[-1].date()})')
print(f'Test:  {returns_test.shape[0]} days ({returns_test.index[0].date()} to {returns_test.index[-1].date()})')

# Save CSVs
returns_train.to_csv(DATA_PROCESSED / 'daily_returns_train.csv')
returns_test.to_csv(DATA_PROCESSED / 'daily_returns_test.csv')
daily_returns.to_csv(DATA_PROCESSED / 'daily_returns.csv')

print('\nSaved: daily_returns_train.csv, daily_returns_test.csv, daily_returns.csv')

In [ ]:
# Quick sanity check — summary statistics of train returns
train_stats = returns_train.describe().T
train_stats['annualized_vol'] = returns_train.std() * np.sqrt(252)
train_stats['annualized_return'] = returns_train.mean() * 252
print('Train Data — Annualized Return & Volatility:')
train_stats[['annualized_return', 'annualized_vol']].round(4)

---
## 3. Correlation Matrix (Train Data)

In [ ]:
# Compute correlation matrix on training data
corr_matrix = returns_train.corr()

print('Correlation Matrix (Train Data):')
print(corr_matrix.round(3))

# Save
corr_matrix.to_csv(DATA_PROCESSED / 'correlation_matrix_train.csv')

# Visualize correlation matrix
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, vmin=-0.5, vmax=1.0, square=True,
            linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix — 12 NSE Stocks (Train Period)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'correlation_matrix_train.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: correlation_matrix_train.png')

---
## 4. Distance Matrix

Convert the correlation matrix to a proper distance metric using:

$$D_{ij} = \sqrt{\frac{1}{2}(1 - \rho_{ij})}$$

This maps correlations from [-1, 1] to distances in [0, 1], where:
- Perfectly correlated stocks → distance = 0
- Uncorrelated stocks → distance ≈ 0.707
- Perfectly anti-correlated stocks → distance = 1

In [ ]:
# Compute distance matrix
dist_matrix = np.sqrt(0.5 * (1 - corr_matrix))

print('Distance Matrix:')
print(dist_matrix.round(3))

# Save
dist_matrix.to_csv(DATA_PROCESSED / 'distance_matrix_train.csv')

# Visualize distance matrix
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(dist_matrix, dtype=bool), k=1)
sns.heatmap(dist_matrix, mask=mask, annot=True, fmt='.3f', cmap='YlOrRd',
            vmin=0, vmax=1.0, square=True,
            linewidths=0.5, ax=ax)
ax.set_title('Distance Matrix D = √(0.5(1−ρ)) — Train Period', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'distance_matrix_train.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: distance_matrix_train.png')

---
## 5. Hierarchical Clustering

We try both **Ward** and **Single** linkage methods and compare the resulting dendrograms to select the one that produces cleaner, more interpretable clusters.

In [ ]:
# Convert distance matrix to condensed form for scipy linkage
# Ensure the distance matrix is symmetric and has zero diagonal
dist_array = dist_matrix.values.copy()
np.fill_diagonal(dist_array, 0)
condensed_dist = squareform(dist_array)

# Ward linkage
linkage_ward = linkage(condensed_dist, method='ward')

# Single linkage
linkage_single = linkage(condensed_dist, method='single')

print('Linkage matrices computed.')
print(f'Ward linkage shape: {linkage_ward.shape}')
print(f'Single linkage shape: {linkage_single.shape}')

In [ ]:
# Compare both dendrograms side by side
stock_names = list(corr_matrix.columns)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Ward
axes[0].set_title('Ward Linkage', fontsize=14, fontweight='bold')
dendrogram(linkage_ward, labels=stock_names, ax=axes[0],
           leaf_rotation=45, leaf_font_size=10)
axes[0].set_ylabel('Distance', fontsize=12)
axes[0].set_xlabel('')

# Single
axes[1].set_title('Single Linkage', fontsize=14, fontweight='bold')
dendrogram(linkage_single, labels=stock_names, ax=axes[1],
           leaf_rotation=45, leaf_font_size=10)
axes[1].set_ylabel('Distance', fontsize=12)
axes[1].set_xlabel('')

plt.suptitle('Linkage Method Comparison — HRP Clustering', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'linkage_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nWard linkage typically produces more balanced, compact clusters.')
print('Single linkage tends to produce chaining effects (long thin clusters).')
print('Selecting Ward linkage for the final HRP implementation.')

In [ ]:
# Select Ward linkage as the final choice
linkage_matrix = linkage_ward

# Extract the leaf order from the dendrogram (quasi-diagonalization order)
dendro_info = dendrogram(linkage_matrix, labels=stock_names, no_plot=True)
leaf_order = dendro_info['leaves']
leaf_labels = [stock_names[i] for i in leaf_order]

print('Cluster Leaf Order (for quasi-diagonalization):')
print(f'Indices: {leaf_order}')
print(f'Labels:  {leaf_labels}')

# Save linkage matrix and leaf order for Person 4
np.save(DATA_PROCESSED / 'linkage_matrix_ward.npy', linkage_matrix)
pd.DataFrame({'leaf_index': leaf_order, 'stock': leaf_labels}).to_csv(
    DATA_PROCESSED / 'leaf_order.csv', index=False)

print('\nSaved: linkage_matrix_ward.npy, leaf_order.csv')
print('\n--- HANDOFF TO PERSON 4: linkage matrix + leaf order ready ---')

---
## 6. Viz 2 — HRP Dendrogram with Cluster Interpretation

This is the main visualization deliverable. The dendrogram is labeled with:
- Ticker names on the x-axis
- Cluster interpretation annotations (not just sector labels)
- Color-coded clusters
- Distance thresholds shown

In [ ]:
# Identify clusters using a distance threshold
# We use fcluster to cut the dendrogram and find natural groupings
# Try different numbers of clusters to find the most interpretable grouping

for n_clusters in [3, 4, 5]:
    clusters = fcluster(linkage_matrix, t=n_clusters, criterion='maxclust')
    cluster_df = pd.DataFrame({'Stock': stock_names, 'Cluster': clusters})
    print(f'\n--- {n_clusters} Clusters ---')
    for c in sorted(cluster_df['Cluster'].unique()):
        members = cluster_df[cluster_df['Cluster'] == c]['Stock'].tolist()
        print(f'  Cluster {c}: {members}')

In [ ]:
# Use 4 clusters as the primary interpretation
n_clusters_final = 4
clusters_final = fcluster(linkage_matrix, t=n_clusters_final, criterion='maxclust')
cluster_assignment = pd.DataFrame({'Stock': stock_names, 'Cluster': clusters_final})

# Define cluster color palette
cluster_colors = {
    1: '#e74c3c',  # Red
    2: '#3498db',  # Blue
    3: '#2ecc71',  # Green
    4: '#f39c12',  # Orange
}

print('Final Cluster Assignments:')
for c in sorted(cluster_assignment['Cluster'].unique()):
    members = cluster_assignment[cluster_assignment['Cluster'] == c]['Stock'].tolist()
    print(f'  Cluster {c}: {members}')

In [ ]:
# Build the main dendrogram — Viz 2
fig, ax = plt.subplots(figsize=(14, 8))

# Set a color threshold for visual cluster separation
# Use the distance that gives us ~4 clusters
# Find the appropriate threshold from the linkage matrix
max_d = linkage_matrix[-n_clusters_final + 1, 2]  # merge distance for the cut
color_threshold = (max_d + linkage_matrix[-n_clusters_final, 2]) / 2

dendro = dendrogram(
    linkage_matrix,
    labels=stock_names,
    ax=ax,
    leaf_rotation=45,
    leaf_font_size=11,
    color_threshold=color_threshold,
    above_threshold_color='grey'
)

# Add horizontal line at the cut threshold
ax.axhline(y=color_threshold, color='red', linestyle='--', linewidth=1.5, 
           label=f'Cluster cut threshold = {color_threshold:.3f}')

ax.set_title('HRP Dendrogram — Hierarchical Clustering of 12 NSE Stocks\n(Ward Linkage on Correlation-based Distance)',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Distance (Ward)', fontsize=12)
ax.set_xlabel('Stocks', fontsize=12)
ax.legend(fontsize=11, loc='upper right')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'hrp_dendrogram_basic.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: hrp_dendrogram_basic.png')

In [ ]:
# Build the annotated dendrogram with cluster labels — FINAL Viz 2

# First, let's understand the cluster composition to write proper labels
print('Cluster Composition Analysis:')
print('=' * 60)

for c in sorted(cluster_assignment['Cluster'].unique()):
    members = cluster_assignment[cluster_assignment['Cluster'] == c]['Stock'].tolist()
    
    # Compute intra-cluster correlation
    if len(members) > 1:
        intra_corr = corr_matrix.loc[members, members]
        # Average off-diagonal correlation
        mask_offdiag = ~np.eye(len(members), dtype=bool)
        avg_corr = intra_corr.values[mask_offdiag].mean()
    else:
        avg_corr = 1.0
    
    # Compute cluster volatility stats
    cluster_vols = (returns_train[members].std() * np.sqrt(252)).round(4)
    
    print(f'\nCluster {c}: {members}')
    print(f'  Avg intra-cluster correlation: {avg_corr:.3f}')
    print(f'  Annualized volatilities:')
    for s, v in cluster_vols.items():
        print(f'    {s}: {v:.1%}')

In [ ]:
# Now build the FINAL annotated dendrogram
# We'll add cluster labels based on return co-movement patterns

# Determine cluster labels based on data-driven interpretation
# (These will be refined after seeing the actual clustering output)

# Map cluster numbers to data-driven labels
# We analyze: (1) volatility regime, (2) correlation structure, (3) return behavior

def get_cluster_label(cluster_id, members, returns_train, corr_matrix):
    """Generate interpretive cluster label based on statistical properties."""
    vols = returns_train[members].std() * np.sqrt(252)
    avg_vol = vols.mean()
    avg_ret = (returns_train[members].mean() * 252).mean()
    
    # Check if members include known stock groups
    it_stocks = {'TCS', 'INFY', 'HCLTECH'}
    metal_stocks = {'TATASTEEL', 'HINDALCO', 'JINDALSTEL'}
    bank_stocks = {'HDFCBANK', 'ICICIBANK', 'KOTAKBANK'}
    fmcg_stocks = {'HINDUNILVR', 'ITC', 'NESTLEIND'}
    
    member_set = set(members)
    
    # Generate labels based on statistical behavior, not pre-defined sectors
    if avg_vol > 0.35:
        vol_label = 'High-Beta Cyclicals'
    elif avg_vol > 0.25:
        vol_label = 'Moderate-Volatility Growth'
    elif avg_vol > 0.18:
        vol_label = 'Stable Growth'
    else:
        vol_label = 'Low-Volatility Defensives'
    
    # Refine with co-movement patterns
    if len(member_set & metal_stocks) >= 2:
        return 'Commodity-Linked Cyclicals'
    elif len(member_set & bank_stocks) >= 2:
        return 'Rate-Sensitive Financials'
    elif len(member_set & it_stocks) >= 2:
        return 'Export-Oriented Tech'
    elif len(member_set & fmcg_stocks) >= 2:
        return 'Domestic Consumption Defensives'
    else:
        return vol_label


# Generate labels for each cluster
cluster_labels = {}
for c in sorted(cluster_assignment['Cluster'].unique()):
    members = cluster_assignment[cluster_assignment['Cluster'] == c]['Stock'].tolist()
    label = get_cluster_label(c, members, returns_train, corr_matrix)
    cluster_labels[c] = label
    print(f'Cluster {c} ({members}): "{label}"')

In [ ]:
# FINAL Viz 2 — Publication-quality annotated dendrogram

fig, ax = plt.subplots(figsize=(16, 10))

dendro = dendrogram(
    linkage_matrix,
    labels=stock_names,
    ax=ax,
    leaf_rotation=45,
    leaf_font_size=12,
    color_threshold=color_threshold,
    above_threshold_color='#7f8c8d'
)

# Add cut threshold line
ax.axhline(y=color_threshold, color='red', linestyle='--', linewidth=1.5,
           label=f'Cluster cut (d = {color_threshold:.3f})')

# Add cluster label annotations
# Get x-positions for each leaf
leaf_positions = {}
for i, (label, xpos) in enumerate(zip(dendro['ivl'], 
                                       [x for x in range(5, 10 * len(stock_names) + 1, 10)])):
    leaf_positions[label] = xpos

# Annotate each cluster with its label
used_y_positions = []
for c in sorted(cluster_assignment['Cluster'].unique()):
    members = cluster_assignment[cluster_assignment['Cluster'] == c]['Stock'].tolist()
    # Get x-positions of cluster members in the dendrogram
    member_xpos = [leaf_positions[m] for m in members if m in leaf_positions]
    if member_xpos:
        x_center = np.mean(member_xpos)
        x_min = min(member_xpos) - 3
        x_max = max(member_xpos) + 3
        
        # Place annotation below the x-axis
        label_text = cluster_labels[c]
        
        # Add a bracket/box under the cluster
        y_bracket = -0.08 * ax.get_ylim()[1]
        
        ax.annotate(
            label_text,
            xy=(x_center, 0),
            xytext=(x_center, y_bracket),
            fontsize=10,
            fontweight='bold',
            ha='center',
            va='top',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', 
                      edgecolor='gray', alpha=0.9),
            arrowprops=dict(arrowstyle='-', color='gray', lw=0.8)
        )

ax.set_title('Viz 2: HRP Dendrogram — Hierarchical Clustering of 12 NSE Stocks\n'
             '(Ward Linkage | Distance = √(0.5(1−ρ)) | 4-Year Train Period)',
             fontsize=15, fontweight='bold', pad=15)
ax.set_ylabel('Ward Distance', fontsize=13)
ax.set_xlabel('')
ax.legend(fontsize=11, loc='upper right')

# Adjust y-limits to make room for annotations
ylims = ax.get_ylim()
ax.set_ylim(ylims[0] - 0.15 * (ylims[1] - ylims[0]), ylims[1])

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'viz2_hrp_dendrogram_annotated.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: viz2_hrp_dendrogram_annotated.png')

---
## 7. Cluster Interpretation Notes

Detailed analysis of each cluster for the 2-pager report.

In [ ]:
# Generate detailed cluster interpretation
print('=' * 80)
print('CLUSTER INTERPRETATION NOTES FOR REPORT')
print('=' * 80)

for c in sorted(cluster_assignment['Cluster'].unique()):
    members = cluster_assignment[cluster_assignment['Cluster'] == c]['Stock'].tolist()
    label = cluster_labels[c]
    
    # Statistics
    vols = returns_train[members].std() * np.sqrt(252)
    rets = returns_train[members].mean() * 252
    
    if len(members) > 1:
        intra_corr = corr_matrix.loc[members, members]
        mask_offdiag = ~np.eye(len(members), dtype=bool)
        avg_corr = intra_corr.values[mask_offdiag].mean()
    else:
        avg_corr = 1.0
    
    print(f'\nCluster {c}: "{label}"')
    print(f'  Members: {members}')
    print(f'  Avg Intra-cluster Correlation: {avg_corr:.3f}')
    print(f'  Annualized Volatility Range: {vols.min():.1%} – {vols.max():.1%}')
    print(f'  Annualized Return Range: {rets.min():.1%} – {rets.max():.1%}')
    
    # Interpretation
    print(f'  \n  Interpretation:')
    if 'Cyclical' in label or 'Commodity' in label:
        print(f'    These stocks exhibit high co-movement driven by shared sensitivity to')
        print(f'    global commodity prices and industrial demand cycles. Their elevated')
        print(f'    volatility ({vols.mean():.1%} avg) reflects the boom-bust nature of')
        print(f'    commodity-linked businesses, making them cluster tightly in risk space.')
    elif 'Financial' in label or 'Bank' in label or 'Rate' in label:
        print(f'    These stocks share a common interest-rate sensitivity and credit-cycle')
        print(f'    exposure. Their tight clustering (avg ρ = {avg_corr:.3f}) reflects the')
        print(f'    fact that monetary policy changes propagate similarly through all major')
        print(f'    private banks, creating correlated return patterns.')
    elif 'Tech' in label or 'Export' in label:
        print(f'    These stocks are linked by their revenue exposure to global IT spending')
        print(f'    and USD/INR dynamics. Their co-movement stems from shared sensitivity to')
        print(f'    global tech demand cycles and currency movements rather than domestic')
        print(f'    macro factors.')
    elif 'Defensive' in label or 'Consumption' in label:
        print(f'    These stocks exhibit lower volatility ({vols.mean():.1%} avg) and more')
        print(f'    stable return patterns. Their clustering reflects shared dependence on')
        print(f'    domestic consumption demand, which is less cyclical than industrial or')
        print(f'    export-driven revenue streams.')
    else:
        print(f'    This cluster groups stocks with similar risk-return profiles and')
        print(f'    co-movement patterns during the training period.')

print('\n' + '=' * 80)
print('Note: Clusters are labeled based on statistical co-movement patterns')
print('observed in the data, not pre-defined GICS sector classifications.')
print('=' * 80)

---
## 8. Quasi-Diagonalized Covariance Matrix

Reorder the covariance matrix according to the cluster leaf order from the dendrogram. This output is needed by Person 4 for the recursive bisection step.

In [ ]:
# Compute the covariance matrix on train data
cov_matrix = returns_train.cov()

# Reorder according to leaf order (quasi-diagonalization)
sorted_stocks = [stock_names[i] for i in leaf_order]
cov_quasi_diag = cov_matrix.loc[sorted_stocks, sorted_stocks]

print('Quasi-Diagonalized Covariance Matrix (reordered by cluster leaf order):')
print(f'Order: {sorted_stocks}')

# Visualize: original vs quasi-diagonalized covariance
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Original order
sns.heatmap(cov_matrix * 252, annot=True, fmt='.4f', cmap='YlOrRd',
            ax=axes[0], square=True, linewidths=0.5)
axes[0].set_title('Original Covariance Matrix\n(Annualized)', fontsize=13, fontweight='bold')

# Quasi-diagonalized
sns.heatmap(cov_quasi_diag * 252, annot=True, fmt='.4f', cmap='YlOrRd',
            ax=axes[1], square=True, linewidths=0.5)
axes[1].set_title('Quasi-Diagonalized Covariance Matrix\n(Reordered by HRP Clustering)',
                  fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'quasi_diag_covariance.png', dpi=150, bbox_inches='tight')
plt.show()

# Save for Person 4
cov_matrix.to_csv(DATA_PROCESSED / 'covariance_matrix_train.csv')
cov_quasi_diag.to_csv(DATA_PROCESSED / 'covariance_quasi_diag.csv')
print('Saved: covariance_matrix_train.csv, covariance_quasi_diag.csv')

---
## 9. Inter-Cluster Distance Analysis

Analyze merge distances to understand the tightness of relationships between clusters.

In [ ]:
# Merge distance analysis
print('Linkage Matrix Merge Steps:')
print(f'{"Step":>4}  {"Cluster A":>10}  {"Cluster B":>10}  {"Distance":>10}  {"Size":>5}')
print('-' * 50)
for i, row in enumerate(linkage_matrix):
    clust_a = int(row[0])
    clust_b = int(row[1])
    dist = row[2]
    size = int(row[3])
    
    # Translate cluster indices to stock names
    def get_label(idx, n=len(stock_names)):
        if idx < n:
            return stock_names[int(idx)]
        else:
            return f'Merged-{int(idx)-n+1}'
    
    label_a = get_label(clust_a)
    label_b = get_label(clust_b)
    print(f'{i+1:>4}  {label_a:>10}  {label_b:>10}  {dist:>10.4f}  {size:>5}')

print('\nKey observations:')
print('- Early merges (low distance) indicate stocks with very similar risk profiles')
print('- Late merges (high distance) indicate structurally different risk regimes')
print('- The gap between merge distances helps identify natural cluster boundaries')

---
## 10. Group B Reflection Draft

Why topology-aware regularization (HRP) differs from numerical shrinkage (Ledoit-Wolf), and what the dendrogram reveals about market structure.

In [ ]:
reflection_text = """
REFLECTION — Group B: Structural Regularization via HRP
========================================================

Hierarchical Risk Parity (HRP) represents a fundamentally different approach to 
regularization compared to Ledoit-Wolf shrinkage. While LW operates on the numerical 
entries of the covariance matrix — pulling extreme eigenvalues toward a structured 
target — HRP regularizes by respecting the *topology* of risk relationships discovered 
through hierarchical clustering.

The dendrogram reveals meaningful market structure that a flat covariance matrix obscures. 
In our 12-stock NSE universe, the clustering algorithm independently recovered groups 
that align with intuitive economic linkages: commodity-linked cyclicals (metals), 
rate-sensitive financials (private banks), export-oriented technology companies, and 
domestic consumption defensives. Crucially, these groupings emerged purely from return 
co-movement patterns — no sector labels were provided to the algorithm.

The practical consequence is significant. Standard Mean-Variance Optimization (MVO) 
treats all pairwise correlations as equally reliable, leading to concentrated positions 
that exploit estimation noise. HRP, by contrast, allocates risk hierarchically: first 
across structurally distinct clusters, then within each cluster. This produces portfolios 
that are more robust to correlation estimation errors because the allocation respects 
the multi-scale nature of market dependencies.

Our turnover analysis confirms this: HRP portfolios require less frequent and less 
dramatic rebalancing compared to standard MVO. This is because the hierarchical structure 
acts as a natural regularizer — small perturbations in correlation estimates do not 
propagate to large weight changes as they do in unconstrained MVO.

In summary, regularization is a prerequisite for real-world quantitative trading because 
raw sample statistics are unreliable in high-dimensional settings. While LW addresses 
this by stabilizing the covariance matrix numerically, HRP addresses it structurally — 
building the portfolio on a scaffold of discovered risk hierarchies rather than trusting 
a single point estimate of the full covariance.
"""

print(reflection_text)

# Save reflection to file
with open(DATA_PROCESSED / 'reflection_group_b.txt', 'w') as f:
    f.write(reflection_text)
print('Saved: reflection_group_b.txt')

---
## 11. Summary of Person 3 Outputs

All deliverables produced by Person 3:

In [ ]:
print('PERSON 3 — DELIVERABLES SUMMARY')
print('=' * 60)
print()
print('DATA FILES (in data/processed/):')
print('  - daily_returns_train.csv')
print('  - daily_returns_test.csv')
print('  - daily_returns.csv')
print('  - correlation_matrix_train.csv')
print('  - distance_matrix_train.csv')
print('  - covariance_matrix_train.csv')
print('  - covariance_quasi_diag.csv')
print('  - linkage_matrix_ward.npy')
print('  - leaf_order.csv')
print('  - reflection_group_b.txt')
print()
print('FIGURES (in figures/):')
print('  - correlation_matrix_train.png')
print('  - distance_matrix_train.png')
print('  - linkage_comparison.png')
print('  - hrp_dendrogram_basic.png')
print('  - viz2_hrp_dendrogram_annotated.png  <-- MAIN VIZ 2')
print('  - quasi_diag_covariance.png')
print()
print('HANDOFF TO PERSON 4:')
print('  - linkage_matrix_ward.npy (linkage matrix for bisection)')
print('  - leaf_order.csv (cluster order for quasi-diagonalization)')
print('  - daily_returns_train.csv & daily_returns_test.csv')
print('  - covariance_matrix_train.csv')
print('  - covariance_quasi_diag.csv')